### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="biogeographical_ancestry_prediction",
    dataset_year="2025", # source data is older, but this version was created in 2025
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="GitHub",
    original_dataset_source_download_link="https://github.com/CarolaHeinzel/BGA_Classification/blob/main/data/input_data/full_data.csv",
    download_description="""
We download the data from the GitHub repository and save it to a predefined folder.

mkdir -p local-data-warehouse/biogeographical_ancestry_prediction/ && curl -L -o local-data-warehouse/biogeographical_ancestry_prediction/filter_population.xlsx https://raw.githubusercontent.com/CarolaHeinzel/BGA-Classification/main/datat/filtered_population_eur_update.xlsx
""",
    # References
    academic_reference_bibtex=r"""@article{heinzel2025advancing,
  title={Advancing biogeographical ancestry predictions through machine learning},
  author={Heinzel, Carola Sophia and Purucker, Lennart and Hutter, Frank and Pfaffelhuber, Peter},
  journal={Forensic Science International: Genetics},
  volume={79},
  pages={103290},
  year={2025},
  publisher={Elsevier}
}
@article{ruiz2023development,
  title={Development and evaluations of the ancestry informative markers of the VISAGE Enhanced Tool for Appearance and Ancestry},
  author={Ruiz-Ram{\'\i}rez, Jorge and de La Puente, M and Xavier, Catarina and Ambroa-Conde, Adri{\'a}n and {\'A}lvarez-Dios, J and Freire-Aradas, A and Mosquera-Miguel, Ana and Ralf, Arwin and Amory, Christina and Katsara, Maria Alexandra and others},
  journal={Forensic Science International: Genetics},
  volume={64},
  pages={102853},
  year={2023},
  publisher={Elsevier}
}
@article{xavier2020development,
  title={Development and validation of the VISAGE AmpliSeq basic tool to predict appearance and ancestry from DNA},
  author={Xavier, Catarina and de la Puente, Maria and Mosquera-Miguel, Ana and Freire-Aradas, Ana and Kalamara, Vivian and Vidaki, Athina and Gross, Theresa E and Revoir, Andrew and Po{\'s}piech, Ewelina and Kartasi{\'n}ska, Ewa and others},
  journal={Forensic Science International: Genetics},
  volume={48},
  pages={102336},
  year={2020},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="heinzel2025advancing,ruiz2023development,xavier2020development",
    license="None", # Data from GitHub does not have a license, source is Apache-2.0 license (I think)
    data_tags=["IID"],
    curation_comments="""
We use this dataset as one of the most recent example of a machine learning task based on the Human Genome project.
We take the targets from the paper by Heinzel et al. (2025) and only rename the targets to be standardized and more concise.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Population",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Population",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_excel(f"{dataset_mold.path}/filter_population.xlsx")
df = df.drop(
    columns=[
        "Unnamed: 1",
        "SNP-Indel with complex genotype listing in VCF (not compiled)",
        "Unnamed: 3",
        "Unnamed: 4",
        "ID",
    ],
)

column_map = {
    "Iberian population in Spain": "Spain (Iberian)",
    "Toscani in Italia": "Italy (Toscani)",
    "Finnish in Finland": "Finland (Finnish)",
    "Utah Residents (CEPH) with N & W European ancestry": "Utah (CEPH, N/W European ancestry)",
    "British in England and Scotland": "UK (England & Scotland, British)",
    "13. Italy - Sardinian": "Italy (Sardinian)",
    "11. France - French": "France (French)",
    "Turkey": "Turkey",
    "09. Russia - Russian": "Russia (Russian)",
    "10. France - French Basque": "France (Basque)"
}
df["Population"] = df["Population"].map(column_map)

for col in df.columns:
    df[col] = df[col].astype("category")
    
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()